# 🤖 Smart Document Assistant — RAG Pipeline & LLM Integration

## Complete End-to-End Implementation

> **What is this project?**  
> A **Retrieval-Augmented Generation (RAG)** system that answers questions from your uploaded documents  
> with high accuracy and minimal hallucinations — grounding every answer in real retrieved evidence.

---

## 🏗️ System Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                  SMART DOCUMENT ASSISTANT PIPELINE                  │
│                                                                     │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌─────────────────┐ │
│  │ Document │──▶│  Text    │──▶│ Chunk    │──▶│  Sentence       │ │
│  │ Loader   │   │ Cleaner  │   │ Splitter │   │  Transformer    │ │
│  └──────────┘   └──────────┘   └──────────┘   │  Embeddings     │ │
│                                                └────────┬────────┘ │
│                                                         │           │
│                                                         ▼           │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌─────────────────┐ │
│  │  Final   │◀──│  OpenAI  │◀──│  Prompt  │◀──│  FAISS Vector   │ │
│  │  Answer  │   │  GPT LLM │   │  Builder │   │  Store + Search │ │
│  └──────────┘   └──────────┘   └──────────┘   └─────────────────┘ │
│                                                         ▲           │
│                                               User Query Embedding  │
└─────────────────────────────────────────────────────────────────────┘
```

## 📋 Table of Contents

1. [Installation & Setup](#1.-Installation-&-Setup)  
2. [Imports & Configuration](#2.-Imports-&-Configuration)  
3. [Document Loading & Preprocessing](#3.-Document-Loading-&-Preprocessing)  
4. [Text Chunking](#4.-Text-Chunking-with-Overlap)  
5. [Embedding Generation](#5.-Embedding-Generation)  
6. [FAISS Vector Store](#6.-FAISS-Vector-Store)  
7. [Semantic Retrieval](#7.-Semantic-Retrieval)  
8. [Prompt Engineering & Guardrails](#8.-Prompt-Engineering-&-Guardrails)  
9. [LLM Integration (OpenAI + LangChain)](#9.-LLM-Integration)  
10. [Evaluation Metrics](#10.-Evaluation-Metrics)  
11. [Visualizations](#11.-Visualizations)  
12. [FastAPI Deployment Simulation](#12.-FastAPI-Deployment)  
13. [Docker Containerization](#13.-Docker-Containerization)  
14. [Summary & Next Steps](#14.-Summary)


---
## 1. Installation & Setup

Install all required libraries. Run this cell once at the start of your session.

| Library | Purpose |
|---|---|
| `langchain` | Orchestration framework — connects all pipeline components |
| `langchain-community` | Community loaders (PDF, text, etc.) |
| `langchain-openai` | OpenAI LLM wrapper for LangChain |
| `sentence-transformers` | Convert text → dense vector embeddings |
| `faiss-cpu` | Fast Approximate Indexing for Similarity Search |
| `openai` | Direct OpenAI API client |
| `pypdf` | Extract text from PDF files |
| `fastapi` + `uvicorn` | Production-style REST API endpoints |
| `matplotlib` + `seaborn` | Visualizations |
| `scikit-learn` | Evaluation metrics + dimensionality reduction |


In [ ]:
!pip install langchain langchain-community langchain-openai sentence-transformers faiss-cpu openai pypdf fastapi uvicorn python-multipart matplotlib seaborn scikit-learn numpy pandas tiktoken --quiet


---
## 2. Imports & Configuration

We import everything we need and configure the OpenAI API key.  
> ⚠️ **Replace `'your-openai-api-key-here'` with your actual key.**  
> You can get one at https://platform.openai.com/api-keys


In [ ]:
import os
import re
import json
import time
import textwrap
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

# ── LangChain ──────────────────────────────────────────────────────────────
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document

# ── Sentence Transformers (local embeddings — no API cost!) ───────────────
from sentence_transformers import SentenceTransformer

# ── FAISS (raw) for custom retrieval ──────────────────────────────────────
import faiss

# ── Scikit-learn for evaluation + visualization ───────────────────────────
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

# ── API Configuration ─────────────────────────────────────────────────────
# Option 1: Set directly (for notebooks)
os.environ['OPENAI_API_KEY'] = 'YOUR_OPEN_API_KEY'

# Option 2: Load from environment (recommended for production)
# import dotenv; dotenv.load_dotenv()  # reads from .env file

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
print('✅ Configuration complete.')
print(f'   OpenAI key loaded: {"YES" if OPENAI_API_KEY and OPENAI_API_KEY != "your-openai-api-key-here" else "⚠️  NOT SET — replace the placeholder above"}')


---
## 3. Document Loading & Preprocessing

The pipeline supports **PDF** and **plain text** files.  
LangChain's `PyPDFLoader` extracts text page-by-page; `TextLoader` reads `.txt` files.

### Why preprocessing matters
Raw extracted text often contains:
- Extra whitespace, line breaks, and control characters
- Page headers/footers (`Page 1 of 10`)
- Hyphenated line breaks (`infor-\nmation` → `information`)

Cleaning this before chunking improves embedding quality significantly.


In [ ]:
# ─── Helper: create a sample document if you don't have a file ────────────
SAMPLE_TEXT = """
Artificial Intelligence and Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn
and improve from experience without being explicitly programmed. It focuses on
developing computer programs that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: The algorithm learns from labeled training data. Examples
   include linear regression, decision trees, and neural networks.

2. Unsupervised Learning: The algorithm finds hidden patterns in unlabeled data.
   Examples include K-means clustering and principal component analysis.

3. Reinforcement Learning: An agent learns by interacting with an environment and
   receiving rewards or penalties for actions.

Deep Learning is a specialized form of machine learning that uses neural networks
with multiple layers (deep neural networks) to model complex patterns.

Natural Language Processing (NLP) is a branch of AI that helps computers understand,
interpret, and generate human language. It powers applications like chatbots,
translation services, and sentiment analysis.

Computer Vision enables machines to interpret and understand visual information
from images and videos. It is used in facial recognition, medical imaging, and
autonomous vehicles.

Retrieval-Augmented Generation (RAG) combines information retrieval with generative
AI. Instead of relying solely on the model's training data, RAG retrieves relevant
documents and injects them into the prompt context for more accurate answers.

Vector databases store high-dimensional numerical representations (embeddings) of
text and enable fast semantic similarity search. Popular vector databases include
FAISS, Pinecone, Weaviate, and ChromaDB.

Transformer models, introduced by the 'Attention is All You Need' paper in 2017,
revolutionized NLP by using self-attention mechanisms to process sequences in parallel.
BERT, GPT, and T5 are prominent transformer architectures.
"""

# Save sample text to file
with open('sample_document.txt', 'w') as f:
    f.write(SAMPLE_TEXT)
print('📄 sample_document.txt created.')


# ─── Document Loader ───────────────────────────────────────────────────────
def load_document(file_path: str) -> list:
    """
    Load a PDF or text document and return a list of LangChain Document objects.
    Each Document has .page_content (text) and .metadata (source, page, etc.)
    """
    ext = os.path.splitext(file_path)[1].lower()
    if ext == '.pdf':
        loader = PyPDFLoader(file_path)
        print(f'📚 Loading PDF: {file_path}')
    elif ext in ['.txt', '.md']:
        loader = TextLoader(file_path, encoding='utf-8')
        print(f'📝 Loading Text File: {file_path}')
    else:
        raise ValueError(f'Unsupported file type: {ext}. Supported: .pdf, .txt')
    
    docs = loader.load()
    print(f'   ✅ Loaded {len(docs)} document section(s)')
    return docs


# ─── Text Cleaner ──────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Remove noise from extracted text:
    - Fix hyphenated line breaks
    - Normalize whitespace
    - Remove non-printable characters
    """
    text = re.sub(r'-(\n)', '', text)          # fix hyphenated line breaks
    text = re.sub(r'\n+', '\n', text)          # collapse multiple newlines
    text = re.sub(r' +', ' ', text)            # collapse multiple spaces
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # remove non-ASCII
    return text.strip()


# ─── Load & Clean ──────────────────────────────────────────────────────────
raw_docs = load_document('sample_document.txt')

# Apply cleaning to each document's page content
for doc in raw_docs:
    doc.page_content = clean_text(doc.page_content)

print(f'\n📊 Document Stats:')
print(f'   Total characters: {sum(len(d.page_content) for d in raw_docs):,}')
print(f'   Total words:      {sum(len(d.page_content.split()) for d in raw_docs):,}')
print(f'\n--- Preview (first 300 chars) ---')
print(raw_docs[0].page_content[:300])


---
## 4. Text Chunking with Overlap

### Why do we chunk?
LLMs have a **context window limit** (e.g. GPT-4 = 128k tokens, but embeddings work best on shorter text).  
Smaller chunks allow:
- **Precise retrieval** — only the relevant portion is fetched
- **Better embeddings** — semantic models work optimally on focused text

### Why overlap?
```
Chunk 1: [===========================|--------]
Chunk 2:                    [--------+=============================]
                            ←overlap→
```
Overlap ensures that sentences/ideas that span chunk boundaries are **not lost**.  
Without overlap, a sentence split across two chunks would be incomplete in both.

### Parameters
| Parameter | Typical Value | Effect |
|---|---|---|
| `chunk_size` | 500–1000 chars | Larger = more context; smaller = more precise |
| `chunk_overlap` | 50–200 chars | Higher = better continuity; more storage |


In [ ]:
# ─── RecursiveCharacterTextSplitter ───────────────────────────────────────
# This splitter tries to split on paragraphs → sentences → words → characters
# preserving semantic structure as much as possible.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,           # max characters per chunk
    chunk_overlap=100,        # overlap between consecutive chunks
    length_function=len,      # measure by character count
    separators=['\n\n', '\n', '. ', ' ', '']  # priority split order
)

chunks = text_splitter.split_documents(raw_docs)

print(f'✅ Chunking complete!')
print(f'   Total chunks created: {len(chunks)}')
print(f'   Avg chunk size:       {np.mean([len(c.page_content) for c in chunks]):.0f} chars')
print(f'   Min chunk size:       {min(len(c.page_content) for c in chunks)} chars')
print(f'   Max chunk size:       {max(len(c.page_content) for c in chunks)} chars')

print('\n--- Sample Chunks ---')
for i, chunk in enumerate(chunks[:3]):
    print(f'\n[Chunk {i+1}] ({len(chunk.page_content)} chars):')
    print(textwrap.fill(chunk.page_content[:200], width=80))
    print('...')

# ─── Chunk Size Distribution Plot ─────────────────────────────────────────
chunk_sizes = [len(c.page_content) for c in chunks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(chunk_sizes, bins=15, color='steelblue', edgecolor='white', alpha=0.85)
ax1.axvline(np.mean(chunk_sizes), color='red', linestyle='--', label=f'Mean: {np.mean(chunk_sizes):.0f}')
ax1.set_title('Chunk Size Distribution', fontsize=13, fontweight='bold')
ax1.set_xlabel('Characters per Chunk')
ax1.set_ylabel('Frequency')
ax1.legend()

ax2.bar(range(len(chunk_sizes)), chunk_sizes, color='steelblue', alpha=0.8)
ax2.axhline(500, color='red', linestyle='--', label='chunk_size=500')
ax2.set_title('Chunk Sizes per Chunk Index', fontsize=13, fontweight='bold')
ax2.set_xlabel('Chunk Index')
ax2.set_ylabel('Characters')
ax2.legend()

plt.tight_layout()
plt.savefig('chunk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Chunk distribution saved.')


---
## 5. Embedding Generation

### What are embeddings?
An **embedding** is a fixed-size numerical vector that encodes the **semantic meaning** of text.  
Text with similar meanings will have vectors that are **geometrically close** in the vector space.

```
  'machine learning'  →  [0.23, -0.11, 0.88, 0.04, ...] (384 dimensions)
  'deep learning'     →  [0.21, -0.09, 0.85, 0.06, ...]  ← similar!
  'cooking recipes'   →  [-0.72, 0.55, -0.13, 0.91, ...] ← very different
```

### The Math: Cosine Similarity

To measure how similar two vectors are, we use **cosine similarity**:

$$\text{cos}(\theta) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \cdot \|\vec{B}\|}$$

Where:
- $\vec{A} \cdot \vec{B}$ = dot product (sum of element-wise products)
- $\|\vec{A}\|$ = L2 norm (magnitude) of vector A
- Result ranges **from -1 to 1**: `1.0` = identical direction, `0` = orthogonal, `-1` = opposite

Cosine similarity is preferred over Euclidean distance because it's **magnitude-invariant** —  
a long sentence and short sentence about the same topic score high even if their vectors have different scales.

### Model: `all-MiniLM-L6-v2`
- Lightweight 22M parameter transformer
- Produces 384-dimensional embeddings
- Fast on CPU, strong semantic quality
- Pretrained on 1B+ sentence pairs


In [ ]:
# ─── Load Embedding Model ─────────────────────────────────────────────────
print('⏳ Loading SentenceTransformer model (all-MiniLM-L6-v2)...')
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Embedding model loaded!')
print(f'   Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}')


# ─── Generate Embeddings for all Chunks ───────────────────────────────────
print('\n⏳ Generating embeddings for all chunks...')
start_time = time.time()

chunk_texts = [c.page_content for c in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # L2 normalize → cosine sim = dot product
)

elapsed = time.time() - start_time
print(f'\n✅ Embeddings generated!')
print(f'   Shape: {chunk_embeddings.shape}  ({chunk_embeddings.shape[0]} chunks × {chunk_embeddings.shape[1]} dims)')
print(f'   Time: {elapsed:.2f}s')


# ─── Cosine Similarity Demo ───────────────────────────────────────────────
print('\n--- Cosine Similarity Demonstration ---')
demo_sentences = [
    'What is machine learning?',
    'Explain deep learning and neural networks.',
    'How do I cook pasta?',
    'What is reinforcement learning?',
]
demo_embeddings = embedding_model.encode(demo_sentences, normalize_embeddings=True)
sim_matrix = cosine_similarity(demo_embeddings)

print('\nPairwise Cosine Similarities:')
short_labels = ['ML question', 'Deep Learning', 'Cooking', 'RL question']
for i in range(len(demo_sentences)):
    for j in range(i+1, len(demo_sentences)):
        print(f'  "{short_labels[i]}" vs "{short_labels[j]}": {sim_matrix[i,j]:.4f}')


---
## 6. FAISS Vector Store

**FAISS** (Facebook AI Similarity Search) is a library for **efficient nearest-neighbor search** in high-dimensional spaces.

### Why not just loop through all vectors?
For N chunks with D-dimensional embeddings, brute-force search is O(N×D).  
FAISS uses **Inverted File Indexes (IVF)** and **Product Quantization** to speed this up dramatically.

### IndexFlatIP — Inner Product Index
Since our embeddings are L2-normalized, **inner product = cosine similarity**.  
We use `IndexFlatIP` (exact search on inner product) — perfect for prototype scale.

```
Query Vector → [FAISS Index] → Top-K most similar chunk vectors → Retrieve chunk texts
```


In [ ]:
# ─── Build FAISS Index ────────────────────────────────────────────────────
dimension = chunk_embeddings.shape[1]  # 384 for all-MiniLM-L6-v2

# IndexFlatIP: exact inner product search (= cosine sim for normalized vectors)
index = faiss.IndexFlatIP(dimension)

# Add all chunk embeddings to the index
index.add(chunk_embeddings.astype('float32'))

print(f'✅ FAISS index built!')
print(f'   Index type:    {type(index).__name__}')
print(f'   Vectors stored: {index.ntotal}')
print(f'   Dimension:      {dimension}')


# ─── Retriever Function ───────────────────────────────────────────────────
def retrieve_chunks(query: str, top_k: int = 3, score_threshold: float = 0.0) -> list:
    """
    Embed the query, search FAISS for top-K most similar chunks.
    Returns list of (chunk_text, similarity_score, chunk_index) tuples.
    """
    # 1. Embed the query
    query_embedding = embedding_model.encode(
        [query], normalize_embeddings=True
    ).astype('float32')
    
    # 2. Search FAISS
    scores, indices = index.search(query_embedding, top_k)
    
    # 3. Collect results
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0 and score >= score_threshold:
            results.append({
                'text': chunk_texts[idx],
                'score': float(score),
                'chunk_index': int(idx)
            })
    return results


# ─── Test Retrieval ───────────────────────────────────────────────────────
test_query = 'What is the difference between supervised and unsupervised learning?'
results = retrieve_chunks(test_query, top_k=3)

print(f'\n🔍 Query: "{test_query}"')
print(f'\n📋 Top-{len(results)} Retrieved Chunks:')
for i, r in enumerate(results):
    print(f'\n  [{i+1}] Score: {r["score"]:.4f} | Chunk #{r["chunk_index"]}')
    print(f'  ' + '-'*60)
    print(textwrap.fill(r['text'][:250], width=72, initial_indent='  '))
    print('  ...')


---
## 7. Semantic Retrieval — Deep Dive

Let's visualize how the retrieval process works by plotting the **cosine similarity scores**  
between our query and all document chunks.


In [ ]:
# ─── Visualize Retrieval Scores ───────────────────────────────────────────
def visualize_retrieval(query: str, top_k: int = 3):
    query_emb = embedding_model.encode([query], normalize_embeddings=True).astype('float32')
    all_scores = cosine_similarity(query_emb, chunk_embeddings)[0]
    
    sorted_idx = np.argsort(all_scores)[::-1]
    sorted_scores = all_scores[sorted_idx]
    
    colors = ['#2ecc71' if i < top_k else '#95a5a6' for i in range(len(sorted_scores))]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(range(len(sorted_scores)), sorted_scores, color=colors, edgecolor='white', linewidth=0.5)
    
    ax.axvline(top_k - 0.5, color='red', linestyle='--', linewidth=1.5, label=f'Top-{top_k} cutoff')
    ax.set_title(f'Retrieval Scores: "{query[:60]}..."', fontsize=12, fontweight='bold')
    ax.set_xlabel('Chunks (sorted by similarity)')
    ax.set_ylabel('Cosine Similarity Score')
    
    green_patch = mpatches.Patch(color='#2ecc71', label=f'Top-{top_k} retrieved')
    grey_patch = mpatches.Patch(color='#95a5a6', label='Not retrieved')
    ax.legend(handles=[green_patch, grey_patch, 
                       plt.Line2D([0],[0], color='red', linestyle='--', label=f'Top-{top_k} cutoff')])
    
    plt.tight_layout()
    plt.savefig('retrieval_scores.png', dpi=150, bbox_inches='tight')
    plt.show()
    return all_scores

scores = visualize_retrieval('What is supervised learning?', top_k=3)
print(f'Max similarity: {scores.max():.4f}')
print(f'Min similarity: {scores.min():.4f}')
print(f'Mean similarity: {scores.mean():.4f}')


---
## 8. Prompt Engineering & Guardrails

A good **prompt template** is critical for RAG quality. We need to:

1. **Inject the retrieved context** into the prompt
2. **Instruct the LLM** to answer ONLY from that context
3. **Add guardrails** to prevent hallucination
4. **Set a fallback** for when context is insufficient

### The RAG Prompt Structure
```
[SYSTEM]  You are a document assistant. Only use the provided context.
[CONTEXT] <retrieved chunk 1>
          <retrieved chunk 2>
          <retrieved chunk 3>
[QUERY]   What is reinforcement learning?
[OUTPUT]  Answer: ...
```

### Guardrail Instructions
| Guardrail | Effect |
|---|---|
| *Answer only from the context below* | Prevents using pretrained knowledge |
| *If not in context, say 'I don't know'* | Prevents hallucination |
| *Cite which part of the context* | Adds grounding and traceability |
| *Do not make up facts* | Explicit prohibition |


In [ ]:
# ─── RAG Prompt Template ──────────────────────────────────────────────────
RAG_PROMPT_TEMPLATE = """\
You are a helpful and precise document assistant. Your job is to answer questions 
based STRICTLY on the context provided below.

RULES:
1. Answer ONLY from the provided context — do NOT use external knowledge.
2. If the answer is not found in the context, respond: "I don't have enough information in the provided documents to answer this question."
3. Be concise and factual.
4. Cite relevant parts of the context when possible.
5. Do NOT make up facts, numbers, or claims.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

# Build a LangChain PromptTemplate object
rag_prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=['context', 'question']
)

print('✅ Prompt template created.')
print('\n--- Template Preview ---')
print(RAG_PROMPT_TEMPLATE[:600])
print('...')


# ─── Build Context from Retrieved Chunks ──────────────────────────────────
def build_context(retrieved_chunks: list) -> str:
    """
    Format retrieved chunks into a single context string for the prompt.
    Each chunk is numbered for traceability.
    """
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(f'[Document Excerpt {i}]\n{chunk["text"].strip()}')
    return '\n\n'.join(context_parts)


# ─── Demo: Full Prompt Construction ───────────────────────────────────────
demo_query = 'What is a transformer model?'
demo_chunks = retrieve_chunks(demo_query, top_k=2)
demo_context = build_context(demo_chunks)
demo_prompt = rag_prompt.format(context=demo_context, question=demo_query)

print(f'\n--- Full Assembled Prompt for: "{demo_query}" ---')
print(demo_prompt[:800])
print(f'\n[Total prompt length: {len(demo_prompt)} characters]')


---
## 9. LLM Integration (OpenAI + LangChain)

We wire everything together using **LangChain's `RetrievalQA` chain**:  
Retriever → Context → Prompt Template → LLM → Final Answer

> ⚠️ **This section requires a valid OpenAI API key.**  
> If you don't have one, the `mock_llm_response()` function below demonstrates the pipeline without API calls.

### LangChain FAISS VectorStore Integration
We'll rebuild the vector store using LangChain's built-in FAISS wrapper for seamless `RetrievalQA` integration.


In [ ]:
# ─── Build LangChain-compatible FAISS Vector Store ─────────────────────────
# LangChain's FAISS wrapper uses OpenAI embeddings by default, but we can
# use a custom Sentence Transformer embedding function.

from langchain.embeddings.base import Embeddings
from typing import List

class SentenceTransformerEmbeddings(Embeddings):
    """Wraps SentenceTransformer to be compatible with LangChain."""
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        embeddings = self.model.encode(texts, normalize_embeddings=True)
        return embeddings.tolist()
    
    def embed_query(self, text: str) -> List[float]:
        embedding = self.model.encode([text], normalize_embeddings=True)
        return embedding[0].tolist()


# Create embedding wrapper
lc_embeddings = SentenceTransformerEmbeddings('all-MiniLM-L6-v2')

# Build LangChain FAISS vector store from our chunks
print('⏳ Building LangChain FAISS vector store...')
vectorstore = FAISS.from_documents(chunks, lc_embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print('✅ Vector store ready!')
print(f'   Indexed {vectorstore.index.ntotal} chunks')


In [ ]:
# ─── RAG Pipeline: With OpenAI API (requires valid API key) ───────────────
def build_rag_pipeline(openai_api_key: str):
    """Build the full RetrievalQA pipeline using OpenAI LLM."""
    llm = ChatOpenAI(
        model_name='gpt-3.5-turbo',
        temperature=0,        # 0 = deterministic, factual
        openai_api_key=openai_api_key,
        max_tokens=500
    )
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type='stuff',          # 'stuff' = paste all context into one prompt
        retriever=retriever,
        chain_type_kwargs={'prompt': rag_prompt},
        return_source_documents=True # return retrieved docs alongside answer
    )
    return qa_chain


# ─── Mock LLM (for demonstration without API key) ─────────────────────────
def mock_llm_response(query: str) -> dict:
    """
    Simulates the RAG pipeline without OpenAI API.
    Shows retrieved context + a placeholder answer.
    """
    retrieved = retrieve_chunks(query, top_k=3)
    context = build_context(retrieved)
    assembled_prompt = rag_prompt.format(context=context, question=query)
    
    mock_answer = (
        f'[MOCK — replace with real LLM call]\n'
        f'Based on the retrieved context, I found {len(retrieved)} relevant sections.\n'
        f'The most relevant chunk (score={retrieved[0]["score"]:.3f}) states:\n'
        f'"{retrieved[0]["text"][:200]}..."'
    )
    
    return {
        'query': query,
        'answer': mock_answer,
        'source_documents': retrieved,
        'context_used': context
    }


# ─── Ask Questions ─────────────────────────────────────────────────────────
TEST_QUERIES = [
    'What is reinforcement learning?',
    'How does BERT differ from GPT?',
    'What is FAISS used for?',
    'What was the year of the Attention paper?',
    'What is the capital of France?',  # out-of-context — should trigger guardrail
]

USE_OPENAI = OPENAI_API_KEY and OPENAI_API_KEY != 'your-openai-api-key-here'

if USE_OPENAI:
    print('🤖 Using real OpenAI GPT pipeline...')
    qa_chain = build_rag_pipeline(OPENAI_API_KEY)

print('\n' + '='*70)
print('RAG QUESTION ANSWERING SESSION')
print('='*70)

qa_log = []
for query in TEST_QUERIES:
    print(f'\n❓ Q: {query}')
    
    if USE_OPENAI:
        result = qa_chain({'query': query})
        answer = result['result']
        sources = result.get('source_documents', [])
    else:
        result = mock_llm_response(query)
        answer = result['answer']
        sources = result['source_documents']
    
    print(f'💬 A: {textwrap.fill(answer, width=70)}')
    print(f'   Sources: {len(sources)} chunks retrieved')
    
    qa_log.append({'query': query, 'answer': answer, 'n_sources': len(sources)})

print('\n' + '='*70)


---
## 10. Evaluation Metrics

How do we know our RAG system is working well? We measure two key metrics:

### 1. Retrieval Precision
$$\text{Precision} = \frac{\text{Relevant chunks retrieved}}{\text{Total chunks retrieved}}$$

A retrieved chunk is **relevant** if it contains keywords from the expected answer.

### 2. Faithfulness Score
$$\text{Faithfulness} = \frac{\text{Answer tokens found in context}}{\text{Total answer tokens}}$$

Measures how much of the generated answer is **grounded** in the retrieved context.  
A faithfulness score close to 1.0 means the model answered from context (not hallucinated).

### 3. Response Latency
Total time from user query to final answer — important for production deployment.


In [ ]:
# ─── Evaluation Dataset ───────────────────────────────────────────────────
eval_dataset = [
    {
        'query': 'What is supervised learning?',
        'expected_keywords': ['labeled', 'training data', 'linear regression', 'algorithm'],
        'answer': 'Supervised learning trains algorithms on labeled training data. Examples include linear regression and decision trees.'
    },
    {
        'query': 'What is RAG?',
        'expected_keywords': ['retrieval', 'generative', 'documents', 'context', 'accurate'],
        'answer': 'RAG combines retrieval with generation to provide more accurate answers by injecting document context.'
    },
    {
        'query': 'What is a transformer model?',
        'expected_keywords': ['attention', 'NLP', '2017', 'BERT', 'GPT', 'self-attention'],
        'answer': 'Transformers use self-attention mechanisms introduced in 2017. BERT and GPT are prominent architectures.'
    },
    {
        'query': 'What does FAISS do?',
        'expected_keywords': ['similarity', 'search', 'vector', 'embeddings', 'nearest'],
        'answer': 'FAISS stores high-dimensional vector embeddings and enables fast similarity search.'
    },
]


# ─── Metric Functions ─────────────────────────────────────────────────────
def compute_retrieval_precision(retrieved_chunks: list, expected_keywords: list) -> float:
    """Fraction of retrieved chunks that contain at least one expected keyword."""
    if not retrieved_chunks:
        return 0.0
    relevant = 0
    for chunk in retrieved_chunks:
        text_lower = chunk['text'].lower()
        if any(kw.lower() in text_lower for kw in expected_keywords):
            relevant += 1
    return relevant / len(retrieved_chunks)


def compute_faithfulness(answer: str, context: str) -> float:
    """Fraction of meaningful answer tokens found in retrieved context."""
    stopwords = {'the','a','an','is','it','in','of','to','and','or','for',
                 'with','on','at','by','from','this','that','was','are','be'}
    answer_tokens = set(re.findall(r'\b\w+\b', answer.lower())) - stopwords
    if not answer_tokens:
        return 0.0
    context_lower = context.lower()
    grounded = sum(1 for token in answer_tokens if token in context_lower)
    return grounded / len(answer_tokens)


# ─── Run Evaluation ───────────────────────────────────────────────────────
print('📊 Running Evaluation...')
print('='*70)

eval_results = []
for item in eval_dataset:
    t0 = time.time()
    retrieved = retrieve_chunks(item['query'], top_k=3)
    latency = time.time() - t0
    
    context = build_context(retrieved)
    precision = compute_retrieval_precision(retrieved, item['expected_keywords'])
    faithfulness = compute_faithfulness(item['answer'], context)
    avg_score = np.mean([r['score'] for r in retrieved]) if retrieved else 0.0
    
    eval_results.append({
        'Query': item['query'][:45] + '...',
        'Precision': precision,
        'Faithfulness': faithfulness,
        'Avg Sim Score': avg_score,
        'Latency (ms)': latency * 1000
    })

eval_df = pd.DataFrame(eval_results)
print(eval_df.to_string(index=False))

print(f'\n--- Aggregate Metrics ---')
print(f'  Mean Retrieval Precision: {eval_df["Precision"].mean():.3f}')
print(f'  Mean Faithfulness:        {eval_df["Faithfulness"].mean():.3f}')
print(f'  Mean Avg Sim Score:       {eval_df["Avg Sim Score"].mean():.3f}')
print(f'  Mean Retrieval Latency:   {eval_df["Latency (ms)"].mean():.1f} ms')


In [ ]:
# ─── Evaluation Visualization ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RAG Pipeline Evaluation Metrics', fontsize=14, fontweight='bold')

query_labels = [f'Q{i+1}' for i in range(len(eval_df))]

# Plot 1: Retrieval Precision
bars1 = axes[0].bar(query_labels, eval_df['Precision'], color='#3498db', edgecolor='white')
axes[0].axhline(y=eval_df['Precision'].mean(), color='red', linestyle='--', label=f'Mean: {eval_df["Precision"].mean():.2f}')
axes[0].set_title('Retrieval Precision', fontweight='bold')
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Precision Score')
axes[0].legend()
for bar, val in zip(bars1, eval_df['Precision']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=11)

# Plot 2: Faithfulness
bars2 = axes[1].bar(query_labels, eval_df['Faithfulness'], color='#2ecc71', edgecolor='white')
axes[1].axhline(y=eval_df['Faithfulness'].mean(), color='red', linestyle='--', label=f'Mean: {eval_df["Faithfulness"].mean():.2f}')
axes[1].set_title('Faithfulness Score', fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Faithfulness Score')
axes[1].legend()
for bar, val in zip(bars2, eval_df['Faithfulness']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=11)

# Plot 3: Latency
bars3 = axes[2].bar(query_labels, eval_df['Latency (ms)'], color='#e74c3c', edgecolor='white')
axes[2].set_title('Retrieval Latency (ms)', fontweight='bold')
axes[2].set_ylabel('Milliseconds')
for bar, val in zip(bars3, eval_df['Latency (ms)']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 11. Visualizations

Visualizing embeddings helps us **understand retrieval behavior** — are semantically  
related chunks actually close together in the vector space?

### Dimensionality Reduction
Our embeddings are 384-dimensional — impossible to visualize directly.  
We use **PCA** (fast, linear) or **t-SNE** (slower, non-linear, better clusters) to project to 2D.

- **PCA**: Finds the two directions of maximum variance. Fast and deterministic.
- **t-SNE**: Preserves local neighborhood structure. Reveals natural clusters better.


In [ ]:
# ─── PCA Visualization ────────────────────────────────────────────────────
print('⏳ Running PCA...')
pca = PCA(n_components=2, random_state=42)
embeddings_2d_pca = pca.fit_transform(chunk_embeddings)
print(f'✅ PCA complete. Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# ─── t-SNE Visualization ──────────────────────────────────────────────────
print('⏳ Running t-SNE (may take a moment)...')
perplexity = min(5, len(chunk_embeddings) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, max_iter=1000)
embeddings_2d_tsne = tsne.fit_transform(chunk_embeddings)
print('✅ t-SNE complete.')


# ─── Plot: PCA + t-SNE side by side ──────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Embedding Space Visualization', fontsize=14, fontweight='bold')

# Query point
test_q_emb = embedding_model.encode(['What is machine learning?'], normalize_embeddings=True)
q_pca = pca.transform(test_q_emb)
q_tsne = tsne.fit_transform(np.vstack([chunk_embeddings, test_q_emb]))[-1:]

# PCA plot
scatter1 = ax1.scatter(embeddings_2d_pca[:, 0], embeddings_2d_pca[:, 1],
                        c=range(len(embeddings_2d_pca)), cmap='viridis',
                        s=80, alpha=0.8, edgecolors='white', linewidth=0.5)
ax1.scatter(q_pca[0, 0], q_pca[0, 1], color='red', s=200, marker='*',
            zorder=5, label='Query point')
for i, (x, y) in enumerate(embeddings_2d_pca):
    ax1.annotate(f'C{i}', (x, y), textcoords='offset points', xytext=(5, 5),
                  fontsize=7, alpha=0.7)
ax1.set_title('PCA — 2D Projection', fontweight='bold')
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax1.legend()
plt.colorbar(scatter1, ax=ax1, label='Chunk Index')

# t-SNE plot
all_tsne = tsne.fit_transform(chunk_embeddings)
scatter2 = ax2.scatter(all_tsne[:, 0], all_tsne[:, 1],
                        c=range(len(all_tsne)), cmap='plasma',
                        s=80, alpha=0.8, edgecolors='white', linewidth=0.5)
for i, (x, y) in enumerate(all_tsne):
    ax2.annotate(f'C{i}', (x, y), textcoords='offset points', xytext=(5, 5),
                  fontsize=7, alpha=0.7)
ax2.set_title('t-SNE — Cluster Structure', fontweight='bold')
ax2.set_xlabel('t-SNE Dimension 1')
ax2.set_ylabel('t-SNE Dimension 2')
plt.colorbar(scatter2, ax=ax2, label='Chunk Index')

plt.tight_layout()
plt.savefig('embedding_visualization.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Similarity Heatmap ────────────────────────────────────────────────────
# Visualize cosine similarity between ALL pairs of chunks
print('⏳ Building similarity heatmap...')
sim_matrix_full = cosine_similarity(chunk_embeddings)

fig, ax = plt.subplots(figsize=(max(8, len(chunks)), max(6, len(chunks)-2)))
mask = np.eye(len(chunks), dtype=bool)  # mask the diagonal (self-similarity = 1.0)

sns.heatmap(
    sim_matrix_full,
    ax=ax,
    cmap='YlOrRd',
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    square=True,
    mask=mask,
    cbar_kws={'label': 'Cosine Similarity'},
    xticklabels=[f'C{i}' for i in range(len(chunks))],
    yticklabels=[f'C{i}' for i in range(len(chunks))]
)

ax.set_title('Inter-Chunk Cosine Similarity Heatmap\n'
             '(high values = semantically related chunks)', fontweight='bold', fontsize=12)
ax.set_xlabel('Chunk Index')
ax.set_ylabel('Chunk Index')
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Heatmap saved. Avg off-diagonal similarity: {sim_matrix_full[~mask].mean():.4f}')


In [ ]:
# ─── Hallucination Comparison Chart ──────────────────────────────────────
# Demonstrate: with vs without RAG, how much faithfulness improves

# Simulated comparison data (based on published RAG benchmarks)
comparison_data = {
    'Metric': ['Factual Accuracy', 'Faithfulness', 'Hallucination Rate',
               'Context Relevance', 'Answer Completeness'],
    'LLM Alone (%)': [62, 48, 38, 55, 70],
    'RAG Pipeline (%)': [89, 92, 8, 91, 85],
}
df_compare = pd.DataFrame(comparison_data)

x = np.arange(len(df_compare))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, df_compare['LLM Alone (%)'], width,
                label='LLM Alone (No RAG)', color='#e74c3c', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, df_compare['RAG Pipeline (%)'], width,
                label='RAG Pipeline', color='#2ecc71', alpha=0.85, edgecolor='white')

ax.set_xlabel('Evaluation Dimension', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('RAG vs No-RAG Performance Comparison\n'
             '(Simulated benchmark — replace with your actual measurements)',
             fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(df_compare['Metric'], rotation=20, ha='right')
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{bar.get_height()}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{bar.get_height()}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('hallucination_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 12. FastAPI Deployment Simulation

We simulate production REST API endpoints using **FastAPI**.  
This makes the RAG pipeline accessible as a microservice.

| Endpoint | Method | Description |
|---|---|---|
| `/` | GET | Health check |
| `/upload` | POST | Upload and index a document |
| `/ask` | POST | Ask a question against indexed documents |
| `/stats` | GET | Index statistics |

> **To run this as a real server**, save the code to `app.py` and run:  
> `uvicorn app:app --host 0.0.0.0 --port 8000 --reload`


In [ ]:
# ─── FastAPI Application Code ──────────────────────────────────────────────
fastapi_code = '''
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import os, time, shutil, tempfile

# Import our pipeline components (from above cells)
# from pipeline import load_document, text_splitter, embedding_model, index, chunk_texts

app = FastAPI(
    title="Smart Document Assistant API",
    description="RAG-powered document question answering",
    version="1.0.0"
)

# ── Shared State ──────────────────────────────────────────────────────────
state = {
    "chunks": [],
    "embeddings": None,
    "index": None,
    "documents_loaded": 0
}

# ── Request/Response Schemas ──────────────────────────────────────────────
class QuestionRequest(BaseModel):
    question: str
    top_k: int = 3

class QuestionResponse(BaseModel):
    question: str
    answer: str
    sources: list
    retrieval_scores: list
    latency_ms: float

# ── Endpoints ─────────────────────────────────────────────────────────────
@app.get("/")
async def root():
    return {"status": "healthy", "service": "Smart Document Assistant", "version": "1.0.0"}

@app.get("/stats")
async def stats():
    return {
        "documents_loaded": state["documents_loaded"],
        "total_chunks": len(state["chunks"]),
        "index_size": state["index"].ntotal if state["index"] else 0
    }

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    if not file.filename.endswith((".pdf", ".txt")):
        raise HTTPException(400, detail="Only .pdf and .txt files are supported.")
    
    with tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(file.filename)[1]) as tmp:
        shutil.copyfileobj(file.file, tmp)
        tmp_path = tmp.name
    
    try:
        docs = load_document(tmp_path)
        new_chunks = text_splitter.split_documents(docs)
        new_texts = [c.page_content for c in new_chunks]
        new_embeddings = embedding_model.encode(new_texts, normalize_embeddings=True)
        
        if state["index"] is None:
            import faiss
            state["index"] = faiss.IndexFlatIP(new_embeddings.shape[1])
        
        state["index"].add(new_embeddings.astype("float32"))
        state["chunks"].extend(new_texts)
        state["documents_loaded"] += 1
        
        return {"message": f"Indexed {len(new_chunks)} chunks from {file.filename}",
                "total_chunks": len(state["chunks"])}
    finally:
        os.unlink(tmp_path)

@app.post("/ask", response_model=QuestionResponse)
async def ask_question(req: QuestionRequest):
    if not state["index"] or len(state["chunks"]) == 0:
        raise HTTPException(400, detail="No documents indexed. Upload a document first via /upload.")
    
    t0 = time.time()
    q_emb = embedding_model.encode([req.question], normalize_embeddings=True).astype("float32")
    scores, indices = state["index"].search(q_emb, req.top_k)
    latency_ms = (time.time() - t0) * 1000
    
    sources = [state["chunks"][i] for i in indices[0] if i >= 0]
    retrieval_scores = [float(s) for s in scores[0]]
    
    # Build context and call LLM (OpenAI or mock)
    context = "\\n\\n".join(f"[Excerpt {i+1}] {s}" for i, s in enumerate(sources))
    answer = f"[Answer based on {len(sources)} retrieved chunks]"  # Replace with LLM call
    
    return QuestionResponse(
        question=req.question,
        answer=answer,
        sources=sources,
        retrieval_scores=retrieval_scores,
        latency_ms=round(latency_ms, 2)
    )
'''

# Save to file
with open('app.py', 'w') as f:
    f.write(fastapi_code)

print('✅ FastAPI application saved to app.py')
print('\n📋 API Endpoint Summary:')
print('   GET  /         → Health check')
print('   GET  /stats    → Index statistics')
print('   POST /upload   → Upload & index a document')
print('   POST /ask      → Ask a question')
print('\n🚀 To run locally:')
print('   uvicorn app:app --host 0.0.0.0 --port 8000 --reload')
print('\n📖 API Docs (auto-generated):')
print('   http://localhost:8000/docs  (Swagger UI)')
print('   http://localhost:8000/redoc (ReDoc)')


---
## 13. Docker Containerization

Docker packages the entire application — code, dependencies, and configuration —  
into a portable **container image** that runs identically on any machine.

```
┌─────────────────────────────────────┐
│          Docker Container           │
│  ┌─────────────────────────────┐   │
│  │  Python 3.11 + Libraries    │   │
│  │  ├── langchain              │   │
│  │  ├── sentence-transformers  │   │
│  │  ├── faiss-cpu              │   │
│  │  └── fastapi + uvicorn      │   │
│  ├─────────────────────────────┤   │
│  │  app.py  (FastAPI app)      │   │
│  └─────────────────────────────┘   │
│  Port 8000 exposed                  │
└─────────────────────────────────────┘
```


In [ ]:
# ─── Generate Dockerfile ──────────────────────────────────────────────────
dockerfile_content = '''# Use official Python 3.11 slim image
FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Copy requirements
COPY requirements.txt .

# Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app.py .

# Expose port
EXPOSE 8000

# Set environment variables
ENV OPENAI_API_KEY=""

# Run FastAPI with Uvicorn
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements_content = '''langchain>=0.2.0
langchain-community>=0.2.0
langchain-openai>=0.1.0
sentence-transformers>=2.7.0
faiss-cpu>=1.7.4
openai>=1.30.0
pypdf>=4.2.0
fastapi>=0.111.0
uvicorn>=0.30.0
python-multipart>=0.0.9
numpy>=1.26.0
pandas>=2.2.0
scikit-learn>=1.5.0
'''

docker_compose_content = '''version: "3.8"
services:
  rag-assistant:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
    volumes:
      - ./uploads:/app/uploads
    restart: unless-stopped
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)
with open('docker-compose.yml', 'w') as f:
    f.write(docker_compose_content)

print('✅ Docker files generated:')
print('   📄 Dockerfile')
print('   📄 requirements.txt')
print('   📄 docker-compose.yml')
print()
print('🐳 To build and run:')
print('   docker build -t rag-assistant .')
print('   docker run -e OPENAI_API_KEY=your-key -p 8000:8000 rag-assistant')
print()
print('   OR with docker-compose:')
print('   docker-compose up --build')


---
## 14. Summary & Next Steps

### ✅ What We Built

| Component | Technology | Role |
|---|---|---|
| Document Loader | LangChain PyPDFLoader / TextLoader | Ingest PDFs & text files |
| Text Cleaner | Regex preprocessing | Remove noise |
| Chunker | RecursiveCharacterTextSplitter | Split into 500-char overlapping chunks |
| Embeddings | SentenceTransformer all-MiniLM-L6-v2 | 384-dim semantic vectors |
| Vector Store | FAISS IndexFlatIP | Fast cosine similarity search |
| Prompt Engineering | LangChain PromptTemplate | Hallucination guardrails |
| LLM | OpenAI GPT via LangChain RetrievalQA | Grounded answer generation |
| Evaluation | Custom precision & faithfulness metrics | Quality measurement |
| API | FastAPI + Uvicorn | Production REST endpoints |
| Deployment | Docker + docker-compose | Portable containerization |

### 🚀 Possible Future Improvements

1. **Scalable Vector Store** — Replace FAISS with Pinecone, Weaviate, or ChromaDB  
2. **Hybrid Retrieval** — Combine semantic search with BM25 keyword search  
3. **Reranking** — Add a cross-encoder reranker (e.g. `cross-encoder/ms-marco-MiniLM-L6-v2`)  
4. **Agentic RAG** — Use LangGraph or LlamaIndex for multi-step reasoning  
5. **Fine-tuned Embeddings** — Domain-specific embedding model improves precision  
6. **Guardrails Framework** — GuardrailsAI or NeMo Guardrails for safety  
7. **Cloud Deployment** — AWS Lambda / GCP Cloud Run / Azure Container Apps  
8. **Streaming Responses** — Server-sent events for real-time answer streaming  

### 📁 Generated Files
```
project/
├── smart_document_assistant.ipynb  ← This notebook
├── app.py                          ← FastAPI application
├── requirements.txt                ← Python dependencies
├── Dockerfile                      ← Docker image definition
├── docker-compose.yml              ← Multi-service orchestration
├── sample_document.txt             ← Sample document for testing
├── chunk_distribution.png          ← Chunk size visualization
├── retrieval_scores.png            ← Retrieval score chart
├── embedding_visualization.png     ← PCA + t-SNE plots
├── similarity_heatmap.png          ← Inter-chunk similarity matrix
└── hallucination_comparison.png    ← RAG vs No-RAG comparison
```

---
*Built as a complete end-to-end RAG system — demonstrating LLM integration,*  
*vector databases, prompt engineering, evaluation, and API deployment.*
